In [36]:
import pandas as pd
from pathlib import Path
import json


In [37]:
repo_root = Path.cwd().parent.parent

domains = pd.read_csv(
    repo_root/"dataset"/"domains.csv.gz", 
    header=None,  
    names=['domain', 'count']  
)

print(domains.head(50))
print(f"Shape: {domains.shape}")


                    domain     count
0              youtube.com  51357326
1                   bit.ly  29985037
2              twitter.com  27442504
3             facebook.com  26765796
4                   goo.gl  22548204
5            instagram.com  15901355
6                 youtu.be  12654279
7                  amzn.to  12161113
8                twitch.tv   5520111
9           soundcloud.com   3997308
10             patreon.com   3899918
11         plus.google.com   3563831
12              discord.gg   1881665
13        itunes.apple.com   1497118
14           pinterest.com   1474366
15             tinyurl.com   1399852
16              amazon.com   1246667
17         play.google.com   1197780
18     creativecommons.org    898587
19               paypal.me    841262
20             smarturl.it    805777
21                   po.st    759743
22      steamcommunity.com    706137
23           teespring.com    668093
24             go.magik.ly    647867
25                on.fb.me    608236
2

In [12]:
# There are 1.2 million urls. Obviously, not all will be relevant

#  Filter domains that appear >= 10 times
domains_filtered_10 = domains[domains['count'] >= 10].copy()
print(f"Domains with >=10 appearances: {len(domains_filtered_10):,}")

#  Filter domains that appear >= 100 times 
domains_filtered_100  = domains[domains['count'] >= 100].copy()
print(f"Domains with >=100 appearances: {len(domains_filtered_100):,}")

#  Filter domains that appear >= 1000 times 
domains_filtered_1000  = domains[domains['count'] >= 1000].copy()
print(f"Domains with >=1000 appearances: {len(domains_filtered_1000):,}")

Domains with >=10 appearances: 206,138
Domains with >=100 appearances: 64,154
Domains with >=1000 appearances: 10,849


In [13]:
domains_filtered_10.to_csv('categorized_domains_10.csv', index=False)
print("Saved categorized domains to 'categorized_domains_10.csv'")

domains_filtered_1000.to_csv('categorized_domains_100.csv', index=False)
print("Saved categorized domains to 'categorized_domains_100.csv'")

domains_filtered_100.to_csv('categorized_domains_1000.csv', index=False)
print("Saved categorized domains to 'categorized_domains_1000.csv'")


Saved categorized domains to 'categorized_domains_10.csv'
Saved categorized domains to 'categorized_domains_100.csv'
Saved categorized domains to 'categorized_domains_1000.csv'


In [14]:
#Iterative analysis - working with >1000 for now

In [38]:
#Initial analysis 
def categorize_domain_initial(domain):
    domain_lower = domain.lower().strip()
    
    # Exact matches 
    exact_matches = {
        # Social Media
        'twitter.com': 'social_media', 'x.com': 'social_media', 'instagram.com': 'social_media',
        'facebook.com': 'social_media', 'tiktok.com': 'social_media', 'linkedin.com': 'social_media',
        'reddit.com': 'social_media', 'discord.com': 'social_media', 'snapchat.com': 'social_media',
        
        # Content Platforms  
        'youtube.com': 'content_platform', 'youtu.be': 'content_platform', 'twitch.tv': 'content_platform',
        'vimeo.com': 'content_platform', 'soundcloud.com': 'content_platform', 'spotify.com': 'content_platform',
        'bandcamp.com': 'content_platform', 'deviantart.com': 'content_platform',
        
        # URL Shorteners
        'bit.ly': 'url_shortener', 'tinyurl.com': 'url_shortener', 'goo.gl': 'url_shortener',
        't.co': 'url_shortener', 'ow.ly': 'url_shortener', 'short.link': 'url_shortener',
        
        # Monetization
        'patreon.com': 'monetization', 'paypal.com': 'monetization', 'ko-fi.com': 'monetization',
        'buymeacoffee.com': 'monetization', 'gofundme.com': 'monetization', 'kickstarter.com': 'monetization',
        'indiegogo.com': 'monetization', 'onlyfans.com': 'monetization',
        
        # E-commerce
        'amazon.com': 'ecommerce', 'ebay.com': 'ecommerce', 'etsy.com': 'ecommerce',
        'aliexpress.com': 'ecommerce', 'alibaba.com': 'ecommerce',
        
        # News/Media
        'bbc.com': 'news_media', 'cnn.com': 'news_media', 'reuters.com': 'news_media',
        'wikipedia.org': 'information', 'wikimedia.org': 'information',
        
        # Gaming
        'steam.com': 'gaming', 'itch.io': 'gaming', 'nexusmods.com': 'gaming',
        
        # Google Services
        'google.com': 'tech_platform', 'gmail.com': 'tech_platform', 'drive.google.com': 'tech_platform',
    }
    
    if domain_lower in exact_matches:
        return exact_matches[domain_lower]
    
    # Pattern matching (medium confidence)
    # Google services
    if 'google.' in domain_lower or domain_lower.endswith('.google.com'):
        return 'tech_platform'
    
    # Social media patterns
    if any(pattern in domain_lower for pattern in ['social', 'chat', 'message']):
        return 'social_media'
    
    # News patterns
    if any(pattern in domain_lower for pattern in ['news', 'times', 'post', 'herald', 'guardian']):
        return 'news_media'
    
    # Blog patterns
    if any(pattern in domain_lower for pattern in ['blog', 'wordpress', 'blogspot', 'medium']):
        return 'blog_personal'
    
    # Gaming patterns
    if any(pattern in domain_lower for pattern in ['game', 'gaming', 'steam', 'play']):
        return 'gaming'
    
    # Shopping patterns
    if any(pattern in domain_lower for pattern in ['shop', 'store', 'buy', 'sell']):
        return 'ecommerce'
    
    # Educational patterns
    if any(pattern in domain_lower for pattern in ['.edu', 'university', 'school', 'learn']):
        return 'education'
    
    # Government patterns
    if any(pattern in domain_lower for pattern in ['.gov', 'government']):
        return 'government'
    
    # Porn patterns (common on internet)
    if any(pattern in domain_lower for pattern in ['porn', 'xxx', 'sex', 'adult']):
        return 'adult_content'
    
    return 'other'


domains_filtered_1000['category'] = domains_filtered_1000['domain'].apply(categorize_domain_initial)

In [39]:
print("=== CATEGORIZATION RESULTS ===")
category_counts = domains_filtered_1000['category'].value_counts()
print(category_counts)

print(f"\n=== COVERAGE ANALYSIS ===")
total_domains = len(domains_filtered_1000)
categorized = len(domains_filtered_1000[domains_filtered_1000['category'] != 'other'])
print(f"Categorized: {categorized:,}/{total_domains:,} ({categorized/total_domains*100:.1f}%)")
print(f"Uncategorized: {total_domains-categorized:,} ({(total_domains-categorized)/total_domains*100:.1f}%)")

=== CATEGORIZATION RESULTS ===
category
other               9561
gaming               341
news_media           259
ecommerce            251
blog_personal        246
education             73
government            41
social_media          32
tech_platform         22
monetization           8
content_platform       6
url_shortener          5
adult_content          4
Name: count, dtype: int64

=== COVERAGE ANALYSIS ===
Categorized: 1,288/10,849 (11.9%)
Uncategorized: 9,561 (88.1%)


In [40]:
# Examine the uncategorized domains 
print("=== TOP UNCATEGORIZED DOMAINS ===")
other_domains = domains_filtered_1000[domains_filtered_1000['category'] == 'other'].sort_values('count', ascending=False)
print("Top 1000 'other' domains to understand patterns:")
print(other_domains.head(1000)[['domain', 'count']])

=== TOP UNCATEGORIZED DOMAINS ===
Top 1000 'other' domains to understand patterns:
                    domain     count
7                  amzn.to  12161113
12              discord.gg   1881665
13        itunes.apple.com   1497118
14           pinterest.com   1474366
18     creativecommons.org    898587
...                    ...       ...
1177              lyft.com      9086
1178               cbn.com      9057
1179            sunnxt.com      9054
1180            mailchi.mp      9047
1181  upload.wikimedia.org      9043

[1000 rows x 2 columns]


In [41]:
# Improved categorization with more patterns
def categorize_domain_improved(domain):
    domain_lower = domain.lower().strip()
    
    exact_matches = {
        # Social Media
        'twitter.com': 'social_media', 'instagram.com': 'social_media', 'facebook.com': 'social_media',
        'reddit.com': 'social_media', 'discord.com': 'social_media', 'linkedin.com': 'social_media',
        'tiktok.com': 'social_media', 'snapchat.com': 'social_media', 'tumblr.com': 'social_media',
        
        # Content Platforms  
        'youtube.com': 'content_platform', 'youtu.be': 'content_platform', 'twitch.tv': 'content_platform',
        'vimeo.com': 'content_platform', 'soundcloud.com': 'content_platform', 'spotify.com': 'content_platform',
        
        # URL Shorteners
        'bit.ly': 'url_shortener', 'tinyurl.com': 'url_shortener', 'goo.gl': 'url_shortener',
        't.co': 'url_shortener', 'ow.ly': 'url_shortener',
        
        # Monetization
        'patreon.com': 'monetization', 'paypal.com': 'monetization', 'ko-fi.com': 'monetization',
        'gofundme.com': 'monetization', 'kickstarter.com': 'monetization',
        
        # E-commerce
        'amazon.com': 'ecommerce', 'ebay.com': 'ecommerce', 'etsy.com': 'ecommerce',
        
        # News/Media - ADD MORE BASED ON YOUR DATA
        'bbc.com': 'news_media', 'cnn.com': 'news_media', 'reuters.com': 'news_media',
        'wikipedia.org': 'information',
        
        # Gaming
        'steam.com': 'gaming',
        
        # Tech
        'google.com': 'tech_platform', 'apple.com': 'tech_platform', 'microsoft.com': 'tech_platform',
    }
    
    if domain_lower in exact_matches:
        return exact_matches[domain_lower]
    
    
    # Country domains (often government/local sites)
    if any(domain_lower.endswith(tld) for tld in ['.gov', '.edu', '.org', '.mil']):
        if '.gov' in domain_lower or '.mil' in domain_lower:
            return 'government'
        elif '.edu' in domain_lower:
            return 'education'
        elif '.org' in domain_lower:
            return 'nonprofit'
    
    # Blog platforms (very common)
    if any(platform in domain_lower for platform in ['wordpress', 'blogspot', 'blogger', 'tumblr', 'medium']):
        return 'blog_personal'
    
    # News patterns (expand)
    if any(pattern in domain_lower for pattern in ['news', 'times', 'post', 'herald', 'guardian', 'daily', 'journal', 'gazette']):
        return 'news_media'
    
    # E-commerce patterns (expand) 
    if any(pattern in domain_lower for pattern in ['shop', 'store', 'buy', 'sell', 'market', 'mall', 'commerce']):
        return 'ecommerce'
    
    # Gaming patterns (expand)
    if any(pattern in domain_lower for pattern in ['game', 'gaming', 'play', 'arcade', 'mmo', 'rpg']):
        return 'gaming'
    
    # Tech/Software patterns
    if any(pattern in domain_lower for pattern in ['tech', 'software', 'app', 'dev', 'code', 'git']):
        return 'tech_platform'
    
    # Adult content patterns
    if any(pattern in domain_lower for pattern in ['porn', 'xxx', 'sex', 'adult', 'cam', 'nude']):
        return 'adult_content'
    
    # Business/Corporate (common pattern)
    if any(pattern in domain_lower for pattern in ['corp', 'inc', 'llc', 'ltd', 'company', 'business']):
        return 'business_corporate'
    
    # Personal websites/portfolios
    if any(pattern in domain_lower for pattern in ['portfolio', 'personal', 'about', 'me', 'bio']):
        return 'personal_website'
    
    return 'other'


In [42]:

# Apply improved categorization
domains_filtered_1000['category_improved'] = domains_filtered_1000['domain'].apply(categorize_domain_improved)

# Compare results
print("=== IMPROVED RESULTS ===")
improved_counts = domains_filtered_1000['category_improved'].value_counts()
print(improved_counts)

improved_categorized = len(domains_filtered_1000[domains_filtered_1000['category_improved'] != 'other'])
print(f"\nImproved coverage: {improved_categorized:,}/{len(domains_filtered_1000):,} ({improved_categorized/len(domains_filtered_1000)*100:.1f}%)")

=== IMPROVED RESULTS ===
category_improved
other                 8035
personal_website       539
blog_personal          428
nonprofit              407
gaming                 353
tech_platform          291
news_media             281
ecommerce              278
adult_content           86
business_corporate      54
government              39
education               35
social_media             8
content_platform         5
url_shortener            5
monetization             5
Name: count, dtype: int64

Improved coverage: 2,814/10,849 (25.9%)


In [44]:
# Examine the uncategorized domain
print("=== TOP UNCATEGORIZED DOMAINS ===")
other_domains = domains_filtered_1000[domains_filtered_1000['category'] == 'other'].sort_values('count', ascending=False)
print("Top 1000 'other' domains to understand patterns:")
print(other_domains.head(1000)[['domain', 'count']])

=== TOP UNCATEGORIZED DOMAINS ===
Top 1000 'other' domains to understand patterns:
                    domain     count
7                  amzn.to  12161113
12              discord.gg   1881665
13        itunes.apple.com   1497118
14           pinterest.com   1474366
18     creativecommons.org    898587
...                    ...       ...
1177              lyft.com      9086
1178               cbn.com      9057
1179            sunnxt.com      9054
1180            mailchi.mp      9047
1181  upload.wikimedia.org      9043

[1000 rows x 2 columns]


In [45]:
def categorize_domain_comprehensive(domain):
    """
    Comprehensive domain categorization based on patterns observed in the dataset
    """
    domain_lower = domain.lower().strip()
    
    # Remove common prefixes for better matching
    clean_domain = domain_lower.replace('www.', '').replace('m.', '').replace('mobile.', '')
    
    # TIER 1: Exact domain matches (highest confidence)
    exact_matches = {
        # Major Social Media Platforms
        'twitter.com': 'social_media',
        'facebook.com': 'social_media',
        'instagram.com': 'social_media',
        'tiktok.com': 'social_media',
        'snapchat.com': 'social_media',
        'linkedin.com': 'social_media',
        'pinterest.com': 'social_media',
        'reddit.com': 'social_media',
        'tumblr.com': 'social_media',
        'myspace.com': 'social_media',
        'vk.com': 'social_media',
        'weibo.com': 'social_media',
        'discord.gg': 'social_media',
        'discord.me': 'social_media',
        'discordapp.com': 'social_media',
        
        # Video/Streaming Platforms
        'youtube.com': 'video_platform',
        'youtu.be': 'video_platform',
        'twitch.tv': 'video_platform',
        'vimeo.com': 'video_platform',
        'dailymotion.com': 'video_platform',
        'liveleak.com': 'video_platform',
        'vine.co': 'video_platform',
        'periscope.tv': 'video_platform',
        'mixer.com': 'video_platform',
        'dlive.tv': 'video_platform',
        
        # Music/Audio Platforms
        'soundcloud.com': 'music_platform',
        'spotify.com': 'music_platform',
        'bandcamp.com': 'music_platform',
        'reverbnation.com': 'music_platform',
        'audiomack.com': 'music_platform',
        'mixcloud.com': 'music_platform',
        
        # URL Shorteners
        'bit.ly': 'url_shortener',
        'goo.gl': 'url_shortener',
        'tinyurl.com': 'url_shortener',
        'ow.ly': 'url_shortener',
        't.co': 'url_shortener',
        'short.link': 'url_shortener',
        'is.gd': 'url_shortener',
        'buff.ly': 'url_shortener',
        'adf.ly': 'url_shortener',
        
        # E-commerce/Marketplace
        'amazon.com': 'ecommerce',
        'ebay.com': 'ecommerce',
        'etsy.com': 'ecommerce',
        'aliexpress.com': 'ecommerce',
        'shopify.com': 'ecommerce',
        'wish.com': 'ecommerce',
        
        # Payment/Monetization
        'patreon.com': 'monetization',
        'paypal.com': 'monetization',
        'paypal.me': 'monetization',
        'ko-fi.com': 'monetization',
        'buymeacoffee.com': 'monetization',
        'gofundme.com': 'monetization',
        'kickstarter.com': 'monetization',
        'indiegogo.com': 'monetization',
        'cash.me': 'monetization',
        'venmo.com': 'monetization',
        
        # Gaming Platforms
        'steam.com': 'gaming',
        'steampowered.com': 'gaming',
        'epicgames.com': 'gaming',
        'roblox.com': 'gaming',
        'minecraft.net': 'gaming',
        'nexusmods.com': 'gaming',
        'curseforge.com': 'gaming',
        'itch.io': 'gaming',
        
        # News/Media Organizations
        'cnn.com': 'news_media',
        'bbc.com': 'news_media',
        'bbc.co.uk': 'news_media',
        'foxnews.com': 'news_media',
        'nytimes.com': 'news_media',
        'wsj.com': 'news_media',
        'reuters.com': 'news_media',
        'theguardian.com': 'news_media',
        
        # Google Services
        'google.com': 'tech_platform',
        'drive.google.com': 'tech_platform',
        'docs.google.com': 'tech_platform',
        'play.google.com': 'tech_platform',
        
        # Apple Services
        'apple.com': 'tech_platform',
        'itunes.apple.com': 'tech_platform',
        'apps.apple.com': 'tech_platform',
        
        # Microsoft Services
        'microsoft.com': 'tech_platform',
        'xbox.com': 'gaming',
        
        # Educational/Reference
        'wikipedia.org': 'educational',
        'wikimedia.org': 'educational',
        'khanacademy.org': 'educational',
    }
    
    if clean_domain in exact_matches:
        return exact_matches[clean_domain]
    
    # TIER 2: Domain extension patterns
    if domain_lower.endswith('.gov'):
        return 'government'
    elif domain_lower.endswith('.edu'):
        return 'educational'
    elif domain_lower.endswith('.org'):
        if any(word in domain_lower for word in ['news', 'media', 'journal']):
            return 'news_media'
        return 'organization'
    elif domain_lower.endswith('.tv'):
        return 'media_content'
    elif domain_lower.endswith('.fm'):
        return 'music_platform'
    
    # TIER 3: Subdomain patterns
    if domain_lower.startswith('shop.') or domain_lower.startswith('store.'):
        return 'ecommerce'
    elif domain_lower.startswith('news.') or domain_lower.startswith('media.'):
        return 'news_media'
    elif domain_lower.startswith('music.') or domain_lower.startswith('play.'):
        return 'music_platform'
    elif domain_lower.startswith('gaming.'):
        return 'gaming'
    
    # TIER 4: Keyword-based categorization
    # Music/Audio keywords
    if any(word in domain_lower for word in ['music', 'audio', 'sound', 'beats', 'records', 'songs', 
                                             'melody', 'rhythm', 'tune', 'mixtape', 'dj', 'radio']):
        return 'music_platform'
    
    # Gaming keywords
    elif any(word in domain_lower for word in ['game', 'gaming', 'gamer', 'play', 'minecraft', 
                                               'fortnite', 'csgo', 'dota', 'lol', 'wow', 'fifa']):
        return 'gaming'
    
    # E-commerce keywords
    elif any(word in domain_lower for word in ['shop', 'store', 'buy', 'market', 'mall', 
                                               'retail', 'sale', 'merch', 'product']):
        return 'ecommerce'
    
    # News/Media keywords
    elif any(word in domain_lower for word in ['news', 'times', 'post', 'herald', 'tribune', 
                                               'gazette', 'journal', 'daily', 'media', 'press']):
        return 'news_media'
    
    # Social/Community keywords
    elif any(word in domain_lower for word in ['social', 'chat', 'talk', 'forum', 'community', 
                                               'connect', 'meet', 'friend']):
        return 'social_media'
    
    # Video/Streaming keywords
    elif any(word in domain_lower for word in ['video', 'stream', 'cast', 'tube', 'flix', 
                                               'watch', 'view', 'tv', 'channel']):
        return 'video_platform'
    
    # Blog/Personal keywords
    elif any(word in domain_lower for word in ['blog', 'wordpress', 'blogspot', 'medium', 
                                               'tumblr', 'diary', 'journal']):
        return 'blog_personal'
    
    # Tech/Software keywords
    elif any(word in domain_lower for word in ['tech', 'soft', 'app', 'code', 'dev', 
                                               'program', 'digital', 'cyber']):
        return 'tech_platform'
    
    # Educational keywords
    elif any(word in domain_lower for word in ['edu', 'learn', 'teach', 'school', 'academy', 
                                               'university', 'college', 'tutorial']):
        return 'educational'
    
    # Sports keywords
    elif any(word in domain_lower for word in ['sport', 'football', 'basketball', 'soccer', 
                                               'nba', 'nfl', 'mlb', 'hockey']):
        return 'sports'
    
    # Finance/Crypto keywords
    elif any(word in domain_lower for word in ['coin', 'crypto', 'bitcoin', 'trade', 'forex', 
                                               'invest', 'finance', 'money']):
        return 'finance_crypto'
    
    # Fashion/Beauty keywords
    elif any(word in domain_lower for word in ['fashion', 'beauty', 'makeup', 'style', 
                                               'cosmetic', 'clothing', 'wear']):
        return 'fashion_beauty'
    
    # Food/Recipe keywords
    elif any(word in domain_lower for word in ['food', 'recipe', 'cook', 'kitchen', 'eat', 
                                               'restaurant', 'pizza', 'burger']):
        return 'food_lifestyle'
    
    # Travel keywords
    elif any(word in domain_lower for word in ['travel', 'tour', 'trip', 'hotel', 
                                               'flight', 'vacation', 'booking']):
        return 'travel'
    
    # Health/Fitness keywords
    elif any(word in domain_lower for word in ['health', 'fitness', 'gym', 'workout', 
                                               'medical', 'doctor', 'hospital']):
        return 'health_fitness'
    
    # Religious keywords
    elif any(word in domain_lower for word in ['church', 'bible', 'islam', 'hindu', 
                                               'buddhist', 'religious', 'faith', 'prayer']):
        return 'religious'
    
    # Adult content patterns (handled carefully)
    elif any(word in domain_lower for word in ['porn', 'xxx', 'adult', 'nsfw']):
        return 'adult_content'
    
    # Regional/Country-specific domains
    elif any(pattern in domain_lower for pattern in ['.co.uk', '.com.au', '.ca', '.de', '.fr', 
                                                     '.es', '.it', '.jp', '.cn', '.in', '.br']):
        return 'regional_site'
    
    # Default category
    return 'other'


domains_filtered_1000['category_improved'] = domains_filtered_1000['domain'].apply(categorize_domain_comprehensive)

print("=== IMPROVED RESULTS ===")
improved_counts = domains_filtered_1000['category_improved'].value_counts()
print(improved_counts)

improved_categorized = len(domains_filtered_1000[domains_filtered_1000['category_improved'] != 'other'])
print(f"\nImproved coverage: {improved_categorized:,}/{len(domains_filtered_1000):,} ({improved_categorized/len(domains_filtered_1000)*100:.1f}%)")



=== IMPROVED RESULTS ===
category_improved
other             6275
video_platform     476
regional_site      473
music_platform     429
news_media         399
organization       392
gaming             380
blog_personal      366
ecommerce          304
media_content      292
tech_platform      270
food_lifestyle     124
educational        120
social_media       113
fashion_beauty     101
sports              83
finance_crypto      76
health_fitness      49
travel              46
government          38
religious           22
monetization        11
url_shortener        8
adult_content        2
Name: count, dtype: int64

Improved coverage: 4,574/10,849 (42.2%)


In [46]:
# Examine the uncategorized domains 
print("=== TOP UNCATEGORIZED DOMAINS ===")
other_domains = domains_filtered_1000[domains_filtered_1000['category'] == 'other'].sort_values('count', ascending=False)
print("Top 1000 'other' domains to understand patterns:")
print(other_domains.head(1000)[['domain', 'count']])

=== TOP UNCATEGORIZED DOMAINS ===
Top 1000 'other' domains to understand patterns:
                    domain     count
7                  amzn.to  12161113
12              discord.gg   1881665
13        itunes.apple.com   1497118
14           pinterest.com   1474366
18     creativecommons.org    898587
...                    ...       ...
1177              lyft.com      9086
1178               cbn.com      9057
1179            sunnxt.com      9054
1180            mailchi.mp      9047
1181  upload.wikimedia.org      9043

[1000 rows x 2 columns]
